# CBCL symptom-onset analysis

This notebook contains the main supervised analysis only. Cohort construction,
participant filtering, covariate matching, and intermediate-file generation are
intentionally excluded.

Required analysis-ready inputs:

- `DATA_ROOT / "code" / "ADHD_dataset.csv"`: feature matrix
- `DATA_ROOT / "code" / "labels.csv"`: one label per row, without a header

In [ ]:
# Portable project paths
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
DATA_ROOT = Path(
    os.environ.get("ANALYSIS_DATA_ROOT", PROJECT_ROOT / "data")
).expanduser().resolve()
RESULTS_DIR = Path(
    os.environ.get("RESULTS_DIR", PROJECT_ROOT / "results")
).expanduser().resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

X = pd.read_csv(DATA_ROOT / "code" / "ADHD_dataset.csv")
y = pd.read_csv(
    DATA_ROOT / "code" / "labels.csv", header=None, names=["label"]
)["label"]

if len(X) != len(y):
    raise ValueError(f"Feature rows ({len(X)}) and labels ({len(y)}) do not match.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    shuffle=True,
    stratify=y,
    random_state=42,
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
import matplotlib.pyplot as plt
import shap
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    learning_rate=0.3,
    n_estimators=55,
    max_depth=1,
    min_child_weight=7,
    subsample=0.96,
    colsample_bytree=1,
    gamma=1.4,
    reg_lambda=15,
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train)

shap.summary_plot(
    shap_values,
    X_train,
    max_display=20,
    feature_names=X_train.columns,
    show=False,
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "shap_summary.pdf", dpi=300, bbox_inches="tight")
plt.show()

shap.summary_plot(
    shap_values,
    X_train,
    max_display=20,
    feature_names=X_train.columns,
    plot_type="bar",
    show=False,
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "shap_bar.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
shap_array = np.asarray(shap_values)
if shap_array.ndim == 2:
    mean_abs_shap = np.abs(shap_array).mean(axis=0)
elif shap_array.shape[-1] == X_train.shape[1]:
    mean_abs_shap = np.abs(shap_array).mean(
        axis=tuple(range(shap_array.ndim - 1))
    )
elif shap_array.ndim == 3 and shap_array.shape[1] == X_train.shape[1]:
    mean_abs_shap = np.abs(shap_array).mean(axis=(0, 2))
else:
    raise ValueError(f"Unexpected SHAP array shape: {shap_array.shape}")

feature_importance = pd.DataFrame(
    {"Feature": X_train.columns, "Mean_SHAP": mean_abs_shap}
).sort_values("Mean_SHAP", ascending=False)
feature_importance.to_csv(
    RESULTS_DIR / "feature_importance.csv", index=False
)

important_features = feature_importance.loc[
    feature_importance["Mean_SHAP"] > 0, "Feature"
].tolist()
if not important_features:
    raise ValueError("SHAP did not select any features.")

X_train_selected = X_train[important_features]
X_test_selected = X_test[important_features]
feature_importance.head(20)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

parameter_grid = {
    "n_estimators": [300, 400, 500],
    "max_features": ["sqrt", "log2", None],
    "max_depth": [5, 6, 7],
    "criterion": ["gini", "entropy"],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    parameter_grid,
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train_selected, y_train)

print(f"Best cross-validation score: {grid_search.best_score_:.3f}")
print(f"Best parameters: {grid_search.best_params_}")

In [ ]:
from sklearn.metrics import (
    RocCurveDisplay,
    classification_report,
    roc_auc_score,
)

rf_model = grid_search.best_estimator_
y_pred = rf_model.predict(X_test_selected)
print(classification_report(y_test, y_pred))

if len(rf_model.classes_) == 2:
    y_score = rf_model.predict_proba(X_test_selected)[:, 1]
    print(f"Test ROC AUC: {roc_auc_score(y_test, y_score):.3f}")
    RocCurveDisplay.from_predictions(y_test, y_score)
    plt.title("Random forest ROC curve")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "roc.pdf", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("ROC plot skipped because the outcome is not binary.")